# 🎙️ VoxCPM2 Khmer — Clone សំឡេងតួប្រុស ៥ នាក់ + រក្សាអារម្មណ៍ដើម
**របៀបប្រើ:** Runtime ➔ Change runtime type ➔ **T4 GPU** ➔ Save ➔ រួចចុច Run ពីលើចុះក្រោមតាមលំដាប់ (Cell 1 ➔ 2 ➔ 3 ➔ 4)

**ធនាគារសំឡេង (Voice Bank):** សំឡេងមិនចងជាមួយតួណាទេ ដាក់ឈ្មោះតាមប្រភេទ `ភេទ-អាយុ-លេខ` ឧ. `male-old-1`, `female-young-2`។ ពេលហៅ API ប្រើ `voice=male-old&character=ជីតា` ➔ ប្រព័ន្ធរើសសំឡេងក្នុងប្រភេទនោះ ហើយតួ "ជីតា" ទទួលសំឡេងដដែលជានិច្ច។ អាច Upload `khmer_voice_bank.zip` ក្នុង Cell 2 ផ្ទាល់។
> ⚠️ ប្រើតែសំឡេងគំរូពី **អ្នកនិយាយខ្មែរកំណើត** ព្រោះសំឡេង clone នឹងយកសំនៀង (accent) តាមគំរូ។

**ឈ្មោះឯកសារសំឡេងគំរូ (Reference):** `ឈ្មោះតួ__អារម្មណ៍.wav`
ឧទាហរណ៍: `male1__normal.wav`, `male1__comedy.wav`, `male2__angry.wav`, `male3__laugh.wav` ...
- បើចង់បានអត្ថបទ (transcript) ជាមួយ ដាក់ឯកសារ `.txt` ឈ្មោះដូចគ្នា: `male1__comedy.txt` ➔ ប្រើ **Ultimate Cloning** (រក្សាសំឡេង + ចង្វាក់ + អារម្មណ៍ត្រង់ជាងគេ)
- សំឡេងគំរូល្អ: ១ នាក់និយាយតែម្នាក់ឯង, ៦–១៥ វិនាទី, គ្មានតន្ត្រី/សំឡេងផ្ទៃខាងក្រោយ, ហើយ **មានអារម្មណ៍ដូចដែលចង់បាន** (កំប្លែង = យកឃ្លាកំប្លែង)

In [ ]:
# ===== CELL 1: ពិនិត្យ GPU + ដំឡើងអ្វីៗទាំងអស់ =====
!nvidia-smi -L
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
!git clone https://github.com/OpenBMB/VoxCPM.git /content/VoxCPM 2>/dev/null || (cd /content/VoxCPM && git pull)
%cd /content/VoxCPM
!pip install -q fastapi uvicorn python-multipart soundfile numpy
!pip install -q einops addict wetext modelscope funasr argbind torchcodec
!pip install -q -e /content/VoxCPM
!pip install -q webrtcvad-wheels librosa
!pip install -q --no-deps resemblyzer
print("✅ ដំឡើងរួចរាល់")

In [ ]:
# ===== CELL 2: បញ្ចូល + រៀបចំសំឡេងគំរូ (តួប្រុស ៥ នាក់) =====
import os, re, sys, shutil, subprocess
from pathlib import Path

# ---- ការកំណត់ ----
USE_GOOGLE_DRIVE = False                    # True = ទាញពី Drive ជំនួសការ Upload
DRIVE_FOLDER     = "/content/drive/MyDrive/VoxCPM_Voices"
CLEAN_VOCALS     = False                    # True = ដកតន្ត្រី/សំឡេងផ្ទៃខាងក្រោយចេញ (Demucs) - ប្រើពេលយកសំឡេងពី video ដើម
BANK_URL         = ""                      # ឬ link ទៅ khmer_voice_bank.zip ក្នុង GitHub (raw ឬ Releases) ➔ ទាញដោយស្វ័យប្រវត្តិ
MAX_REF_SECONDS  = 15                       # កាត់សំឡេងគំរូមិនឱ្យលើសនេះ (ពេលគ្មាន .txt)

RAW_DIR   = Path("/content/voices_raw")
VOICE_DIR = Path("/content/voices")
RAW_DIR.mkdir(exist_ok=True, parents=True)
VOICE_DIR.mkdir(exist_ok=True, parents=True)

AUDIO_EXT = {".wav",".mp3",".m4a",".flac",".ogg",".aac",".mp4",".mkv",".mov",".webm"}

def _safe(s):
    return re.sub(r"[^a-z0-9_\-]", "_", s.strip().lower()) or "voice"

def _duration(path):
    try:
        out = subprocess.run(["ffprobe","-v","error","-show_entries","format=duration","-of","csv=p=0",str(path)],
                             capture_output=True, text=True).stdout.strip()
        return float(out)
    except Exception:
        return 0.0

def extract_vocals(path):
    tmp = Path("/content/_tmp_in.wav")
    subprocess.run(["ffmpeg","-y","-loglevel","error","-i",str(path),"-vn","-ac","2","-ar","44100",str(tmp)], check=True)
    out = Path("/content/demucs_out"); shutil.rmtree(out, ignore_errors=True)
    subprocess.run([sys.executable,"-m","demucs","--two-stems=vocals","-o",str(out),str(tmp)], check=True)
    return next(out.rglob("vocals.wav"))

def process_reference(src, voice, emotion="normal", text="", max_sec=MAX_REF_SECONDS, clean=False):
    # แปลงเป็น mono 48kHz, កាត់ស្ងាត់ដើម, ធ្វើឱ្យកម្រិតសំឡេងស្មើគ្នា
    voice, emotion = _safe(voice), _safe(emotion)
    dst_dir = VOICE_DIR / voice; dst_dir.mkdir(exist_ok=True, parents=True)
    dst = dst_dir / f"{emotion}.wav"
    if clean:
        src = extract_vocals(src)
    cmd = ["ffmpeg","-y","-loglevel","error","-i",str(src),"-vn","-ac","1","-ar","48000",
           "-af","silenceremove=start_periods=1:start_threshold=-45dB:start_silence=0.1,loudnorm=I=-20:TP=-2:LRA=11"]
    if not text:
        cmd += ["-t", str(max_sec)]       # មានអត្ថបទ = មិនកាត់ (ក្រែងអត្ថបទលែងត្រូវនឹងសំឡេង)
    cmd.append(str(dst))
    subprocess.run(cmd, check=True)
    if text:
        dst.with_suffix(".txt").write_text(text.strip(), encoding="utf-8")
        if _duration(dst) > 20:
            print(f"⚠️ {voice}/{emotion}: សំឡេងវែងជាង 20 វិនាទី ➔ គួរកាត់ឱ្យខ្លី (6–15s) ហើយ transcript ត្រូវត្រូវគ្នា ១០០%")
    elif dst.with_suffix(".txt").exists():
        dst.with_suffix(".txt").unlink()
    return dst

# ---- ទទួលឯកសារ ----
if BANK_URL:
    subprocess.run(["wget", "-q", "-O", str(RAW_DIR / "voice_bank.zip"), BANK_URL], check=True)
    print("⬇️ ទាញធនាគារសំឡេងពី GitHub រួចរាល់")
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    for f in Path(DRIVE_FOLDER).glob("*"):
        shutil.copy(f, RAW_DIR / f.name)
elif not any(RAW_DIR.iterdir()):
    from google.colab import files
    print("📤 ជ្រើសរើសឯកសារសំឡេងគំរូ (+ ឯកសារ .txt បើមាន) ទាំងអស់ក្នុងពេលតែមួយ")
    for name, data in files.upload().items():
        (RAW_DIR / name).write_bytes(data)
else:
    print("ℹ️ មានឯកសារក្នុង", RAW_DIR, "រួចហើយ (លុបវាបើចង់ Upload ថ្មី)")

# ---- ធនាគារសំឡេងដែលរៀបចំរួច (.zip) ----
for z in sorted(RAW_DIR.glob("*.zip")):
    tmp = Path("/content/_bank_tmp"); shutil.rmtree(tmp, ignore_errors=True)
    shutil.unpack_archive(str(z), str(tmp))
    for d in tmp.iterdir():
        if d.is_dir() and d.name != "__MACOSX":
            shutil.copytree(d, VOICE_DIR / d.name, dirs_exist_ok=True)
    print("📦 ផ្ទុកធនាគារសំឡេងពី", z.name)

def f0_hint(path):
    # ប៉ាន់ស្មាន pitch (Hz) ដើម្បីជួយបែងចែកប្រុស/ស្រី — មិនអាចប្រាប់អាយុ ឬភាសាបានទេ
    try:
        import numpy as np
        from scipy.io import wavfile
        from scipy.signal import fftconvolve
        subprocess.run(["ffmpeg","-y","-loglevel","error","-i",str(path),"-ac","1","-ar","16000","/content/_f0.wav"], check=True)
        sr, x = wavfile.read("/content/_f0.wav"); x = x.astype("float32"); x /= (abs(x).max() + 1e-9)
        n, h = int(0.04*sr), int(0.01*sr); lo, hi = int(sr/400), int(sr/70); out = []
        for i in range(0, len(x) - n, h):
            f = x[i:i+n] * np.hanning(n)
            if (f**2).mean()**0.5 < 0.01: continue
            ac = fftconvolve(f, f[::-1])[n-1:]; ac /= ac[0] + 1e-9
            k = int(np.argmax(ac[lo:hi]))
            if ac[lo+k] > 0.45: out.append(sr / (lo + k))
        return int(np.median(out)) if out else None
    except Exception:
        return None

# ---- ដំណើរការ ----
texts = {p.stem: p.read_text(encoding="utf-8").strip() for p in RAW_DIR.glob("*.txt")}
done = []
for f in sorted(RAW_DIR.iterdir()):
    if f.suffix.lower() not in AUDIO_EXT:
        continue
    stem = f.stem
    voice, emotion = stem.split("__", 1) if "__" in stem else (stem, "normal")
    process_reference(f, voice, emotion, texts.get(stem, ""), clean=CLEAN_VOCALS)
    done.append((_safe(voice), _safe(emotion), "✅ មាន transcript (Ultimate)" if stem in texts else "➖ គ្មាន transcript (Clone ធម្មតា)"))

print(f"\n🎭 បានរៀបចំ {len(done)} ឯកសារ:")
for v, e, t in done:
    print(f"  • {v:16s} / {e:10s} F0≈{f0_hint(VOICE_DIR / v / (e + '.wav'))} Hz  {t}")
n_voices = len([d for d in VOICE_DIR.iterdir() if d.is_dir()])
if n_voices != 5:
    print(f"\nℹ️ ឥឡូវមាន {n_voices} សំឡេងក្នុងធនាគារ — អាចបន្ថែមក្រោយតាម API ឬ Upload ម្តងទៀត")

In [ ]:
# ===== CELL 3: ផ្ទុកម៉ូដែល + មុខងារបង្កើតសំឡេង + សាកល្បងស្តាប់ =====
import sys, re, time, inspect, threading
sys.path.insert(0, "/content/VoxCPM/src"); sys.path.insert(0, "/content/VoxCPM")
import numpy as np, torch, soundfile as sf
import voxcpm

RUN_AUDITION = True
TEST_TEXT = "សួស្តី! នេះគឺជាការសាកល្បងសំឡេងរបស់ខ្ញុំ។ ហាហា ពិតជាសប្បាយណាស់!"

device = "cuda" if torch.cuda.is_available() else "cpu"
if "model" not in globals() or model is None:
    try:
        model = voxcpm.VoxCPM.from_pretrained("openbmb/VoxCPM2", device=device, optimize=True)
    except Exception:
        model = voxcpm.VoxCPM.from_pretrained("openbmb/VoxCPM2", device=device)
SR = getattr(getattr(model, "tts_model", None), "sample_rate", 48000)
print(f"✅ VoxCPM2 រួចរាល់លើ {device.upper()} | {SR} Hz")

# ---- បញ្ជីសំឡេង ----
VOICES = {}
def scan_voices():
    VOICES.clear()
    for vd in sorted(VOICE_DIR.iterdir()):
        if not vd.is_dir(): continue
        for wav in sorted(vd.glob("*.wav")):
            txt = wav.with_suffix(".txt")
            VOICES.setdefault(vd.name, {})[wav.stem] = {
                "wav": str(wav),
                "text": txt.read_text(encoding="utf-8").strip() if txt.exists() else ""}
scan_voices()

import json, collections
ASSIGN_PATH = VOICE_DIR / "assignments.json"
ASSIGN = json.loads(ASSIGN_PATH.read_text(encoding="utf-8")) if ASSIGN_PATH.exists() else {}

def pick_ref(voice, emotion="", character=""):
    # voice អាចជាឈ្មោះពេញ (male-old-1) ឬ tag (male-old, male, female)
    q = voice.strip().lower()
    cands = [q] if q in VOICES else sorted(v for v in VOICES if v.startswith(q))
    if not cands:
        raise KeyError(f"មិនមានសំឡេង/tag '{voice}' — មាន: {list(VOICES)}")
    chosen = cands[0]
    ch = character.strip().lower()
    if ch and len(cands) > 1:
        key = f"{q}|{ch}"
        if ASSIGN.get(key) in cands:
            chosen = ASSIGN[key]
        else:
            used = collections.Counter(ASSIGN.values())
            chosen = min(cands, key=lambda v: (used[v], cands.index(v)))   # ជ្រើសសំឡេងដែលមិនទាន់ត្រូវបានប្រើ
            ASSIGN[key] = chosen
            ASSIGN_PATH.write_text(json.dumps(ASSIGN, ensure_ascii=False, indent=1), encoding="utf-8")
    v = VOICES[chosen]
    e = emotion.strip().lower()
    if e not in v:
        e = "normal" if "normal" in v else next(iter(v))
    return v[e]["wav"], v[e]["text"], e, chosen

def khmer_ratio(text):
    letters = [c for c in text if c.isalpha()]
    return (sum(1 for c in letters if "\u1780" <= c <= "\u17ff") / len(letters)) if letters else 1.0

# ---- តម្រង parameter ដែល model.generate ទទួលស្គាល់ ----
_sig = inspect.signature(model.generate)
_ACCEPT = None if any(p.kind == p.VAR_KEYWORD for p in _sig.parameters.values()) else set(_sig.parameters)

def _split(text, max_chars=90):
    parts = [p.strip() for p in re.split(r"(?<=[។៕!?.\n])", text) if p.strip()]
    out, cur = [], ""
    for p in parts:
        if cur and len(cur) + len(p) > max_chars:
            out.append(cur); cur = p
        else:
            cur = (cur + " " + p).strip()
    if cur: out.append(cur)
    return out or [text.strip()]

GEN_LOCK = threading.Lock()
DEFAULT_SEED = 1234        # ប្តូរលេខនេះបើមិនពេញចិត្តសំឡេង
SPEAKER_CHECK = True       # ត្រួតពិនិត្យថាសំឡេងដូចគំរូឬអត់
SIM_THRESHOLD = 0.80       # ភាពដូចអប្បបរមា (0–1) ទាបពេក = ឆាប់ទទួលយក, ខ្ពស់ពេក = ព្យាយាមច្រើនដង
DEFAULT_TRIES = 4          # ព្យាយាមបង្កើតម្នាក់ប្រយោគអតិបរមាប៉ុន្មានដង

# ---- ឆ្មាំត្រួតពិនិត្យស្នាមសំឡេង (Resemblyzer) ----
encoder = None
if SPEAKER_CHECK:
    try:
        from resemblyzer import VoiceEncoder, preprocess_wav
        encoder = VoiceEncoder(device=device)
        print("🛡️ Speaker check បើករួចរាល់")
    except Exception as e:
        print("⚠️ បើក Speaker check មិនបាន ➔ រត់ដោយគ្មានវា:", e)

_REF_EMB = {}
def _ref_embedding(path):
    if path not in _REF_EMB:
        _REF_EMB[path] = encoder.embed_utterance(preprocess_wav(path))
    return _REF_EMB[path]

def _similarity(w, ref_emb):
    # ប្រយោគខ្លីពេក (<1.2 វិនាទី) វាស់មិនសូវត្រូវ ➔ មិនបដិសេធ
    if len(w) < SR * 1.2:
        return None
    try:
        e = encoder.embed_utterance(preprocess_wav(w, source_sr=SR))
        return float(np.dot(e, ref_emb))
    except Exception:
        return None

LAST_REPORT = {"sims": [], "min": None}

def generate_speech(text, ref=None, prompt_text="", style="", mode="auto",
                    timesteps=20, cfg=2.0, seed=-1, denoise=False, tries=DEFAULT_TRIES):
    # mode: auto | ultimate (ត្រូវការ prompt_text) | clone (ប្រើ style បាន)
    if ref and mode in ("auto", "ultimate") and prompt_text and not style:
        used = "ultimate"
    elif ref:
        used = "clone"
    else:
        used = "design"
    chunks, wavs, sims = _split(text), [], []
    gap = np.zeros(int(SR * 0.12), dtype=np.float32)
    ref_emb = _ref_embedding(ref) if (encoder is not None and ref) else None
    n_try = max(1, tries) if ref_emb is not None else 1
    with GEN_LOCK:
        base_seed = seed if seed >= 0 else DEFAULT_SEED
        for c in chunks:
            best, best_s = None, -1.0
            for t in range(n_try):
                torch.manual_seed(base_seed + t * 101)   # ព្យាយាមដំបូង = seed មូលដ្ឋាន (ថេរ)
                kw = dict(text=c, cfg_value=cfg, inference_timesteps=timesteps, normalize=True,
                          denoise=denoise, retry_badcase=True)
                if used == "ultimate":
                    kw.update(prompt_wav_path=ref, prompt_text=prompt_text, reference_wav_path=ref)
                else:
                    if used == "clone":
                        kw["reference_wav_path"] = ref
                    if style:
                        kw["text"] = f"({style}){c}"
                if _ACCEPT is not None:
                    kw = {k: v for k, v in kw.items() if k in _ACCEPT}
                w = np.asarray(model.generate(**kw), dtype=np.float32).squeeze()
                sim = _similarity(w, ref_emb) if ref_emb is not None else None
                if sim is None:                      # វាស់មិនបាន ➔ ទទួលយក
                    best, best_s = w, None
                    break
                if sim > best_s:
                    best, best_s = w, sim
                if sim >= SIM_THRESHOLD:             # ដូចគ្រប់គ្រាន់ ➔ ឈប់
                    break
            if best_s is not None:
                sims.append(round(best_s, 3))
            wavs += [best, gap]
    LAST_REPORT["sims"] = sims
    LAST_REPORT["min"] = min(sims) if sims else None
    return np.concatenate(wavs[:-1]), used

# ---- សាកល្បងស្តាប់ ----
if RUN_AUDITION:
    from IPython.display import Audio, display
    for v, emos in VOICES.items():
        for e, info in emos.items():
            t0 = time.time()
            w, used = generate_speech(TEST_TEXT, info["wav"], info["text"])
            print(f"🎧 {v} / {e}  [{used}]  {time.time()-t0:.1f}s  ភាពដូចអប្បបរមា={LAST_REPORT['min']}")
            display(Audio(w, rate=SR))

In [ ]:
# ===== CELL 4: បើក API Server + Public URL (Cell នេះនឹងរត់ជាប់រហូត) =====
import os, re, time, uuid, subprocess, threading, shutil
import uvicorn
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.responses import FileResponse, JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from starlette.concurrency import run_in_threadpool

os.makedirs("/content/uploads", exist_ok=True)
os.makedirs("/content/outputs", exist_ok=True)

app = FastAPI(title="VoxCPM2 Khmer Multi-Voice API")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True,
                   allow_methods=["*"], allow_headers=["*"], expose_headers=["X-Mode","X-Voice","X-Emotion","X-Min-Similarity","X-Warning"])

@app.get("/")
def home():
    return {"status": "ok", "model": "VoxCPM2", "device": device, "voices": {v: list(e) for v, e in VOICES.items()}}

@app.get("/api/voices")
def list_voices():
    return {v: {e: {"has_transcript": bool(i["text"])} for e, i in emos.items()} for v, emos in VOICES.items()}

@app.get("/api/assignments")
def list_assignments():
    return ASSIGN

@app.post("/api/voices/register")
async def register_voice(voice: str = Form(...), emotion: str = Form("normal"),
                         transcript: str = Form(""), audio: UploadFile = File(...)):
    tmp = f"/content/uploads/reg_{uuid.uuid4().hex}_{os.path.basename(audio.filename or 'a.wav')}"
    with open(tmp, "wb") as f:
        f.write(await audio.read())
    try:
        await run_in_threadpool(process_reference, tmp, voice, emotion, transcript)
        scan_voices()
    except Exception as e:
        raise HTTPException(500, str(e))
    return {"ok": True, "voices": {v: list(e) for v, e in VOICES.items()}}

# ត្រូវគ្នានឹង Web Studio ចាស់ (text + reference_audio) ហើយបន្ថែមជម្រើសថ្មី
@app.post("/api/clone-and-speak")
async def clone_and_speak(
    text: str = Form(...),
    reference_audio: UploadFile = File(None),
    voice: str = Form(""),          # ឈ្មោះសំឡេង ឬ tag ដូច male-old, female
    character: str = Form(""),      # ឈ្មោះតួក្នុងរឿង ➔ ទទួលបានសំឡេងដដែលជានិច្ច
    emotion: str = Form(""),        # normal / comedy / angry ...
    style: str = Form(""),          # ឧ. "comedic, exaggerated, playful" (ប្រើរបៀប clone)
    prompt_text: str = Form(""),    # transcript នៃ reference_audio (ដើម្បី Ultimate)
    mode: str = Form("auto"),
    timesteps: int = Form(20),
    cfg_value: float = Form(2.0),
    seed: int = Form(-1),
    denoise: bool = Form(False),
    tries: int = Form(4),           # ចំនួនដងអតិបរមាក្នុងការរើសសំឡេងដែលដូចបំផុត
):
    try:
        ref, ptxt, emo = None, prompt_text.strip(), ""
        if reference_audio is not None and reference_audio.filename:
            ref = f"/content/uploads/{uuid.uuid4().hex}_{os.path.basename(reference_audio.filename)}"
            with open(ref, "wb") as f:
                f.write(await reference_audio.read())
            if os.path.getsize(ref) < 1000:
                ref = None
        elif voice.strip():
            try:
                ref, ptxt, emo, voice = pick_ref(voice, emotion, character)
            except KeyError as e:
                raise HTTPException(404, str(e))
        wav, used = await run_in_threadpool(
            generate_speech, text.strip(), ref, ptxt, style.strip(), mode, timesteps, cfg_value, seed, denoise, tries)
        out = f"/content/outputs/out_{int(time.time()*1000)}_{uuid.uuid4().hex[:6]}.wav"
        sf.write(out, wav, SR)
        return FileResponse(out, media_type="audio/wav",
                            headers={"X-Mode": used, "X-Voice": voice, "X-Emotion": emo, "X-Min-Similarity": str(LAST_REPORT["min"]),
                                     "X-Warning": "non-khmer-text" if khmer_ratio(text) < 0.6 else ""})
    except HTTPException:
        raise
    except Exception as e:
        print("❌", e)
        raise HTTPException(500, str(e))

threading.Thread(target=lambda: uvicorn.Server(
    uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning")).run(), daemon=True).start()
time.sleep(3)

print("🌐 កំពុងបើក Cloudflare Tunnel...")
log_path = "/content/cloudflared.log"
with open(log_path, "w") as lf:
    subprocess.Popen(["cloudflared","tunnel","--url","http://127.0.0.1:8000"], stdout=lf, stderr=lf)
public_url = None
for _ in range(60):
    time.sleep(0.5)
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", open(log_path, errors="ignore").read())
    if m:
        public_url = m.group(0); break

print("\n" + "=" * 66)
if public_url:
    print("🎉 PUBLIC URL:", public_url)
    print("👉 ចម្លងទៅដាក់ក្នុង Web Studio ➔ Settings ➔ VoxCPM2 Server URL")
    print("   តួដែលមាន:", {v: list(e) for v, e in VOICES.items()})
else:
    print("⚠️ មិនទាន់ឃើញ URL — មើល /content/cloudflared.log")
print("=" * 66)
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("🛑 Stopped")